## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

### **Transformers**


##### **Tarea 1: predicción veracidad**


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------------------------------------
# Configuración Global
# -------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 10         # Aumentado a 10 como pediste
PATIENCE = 3        # Early Stopping: Si no mejora en 3 épocas, paramos
LR = 2e-5
MAX_LEN = 128
DATA_PATH = "data/train_preprocessed.parquet"

models_to_compare = [
    "distilbert-base-uncased",
    "distilroberta-base"
]

print(f"Usando dispositivo: {DEVICE}")

# -------------------------------------
# 1. Carga y Balanceo de Datos (Oversampling)
# -------------------------------------
df = pd.read_parquet(DATA_PATH)
df = df[df["sender_labels"].astype(str).str.lower().isin(["true", "false"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0}).values

le = LabelEncoder()
df['target'] = le.fit_transform(y_raw)

# Split
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    df["messages"], df["target"], stratify=df["target"], test_size=0.2, random_state=42
)

# Oversampling en Train
train_df = pd.DataFrame({'text': X_train_raw, 'label': y_train_raw})
df_false = train_df[train_df['label'] == 0]
df_true = train_df[train_df['label'] == 1]

# Igualamos False a True
df_false_over = df_false.sample(len(df_true), replace=True, random_state=42)
df_balanced = pd.concat([df_true, df_false_over], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

X_train = df_balanced['text'].values
y_train = df_balanced['label'].values
X_val = X_val_raw.values
y_val = y_val.values

print(f"Datos preparados. Train (Balanceado): {len(X_train)} | Val: {len(X_val)}")

# -------------------------------------
# 2. Clases Utilitarias
# -------------------------------------
class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def train_model(model_name, X_train, y_train, X_val, y_val):
    print(f"\n{'='*40}")
    print(f"PROCESANDO: {model_name}")
    print(f"{'='*40}")
    
    # Cargar Tokenizer y Modelo específicos
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to(DEVICE)
    
    # Dataloaders
    train_ds = TransformerDataset(X_train, y_train, tokenizer, MAX_LEN)
    val_ds = TransformerDataset(X_val, y_val, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    
    # Variables de control
    best_mcc = -1
    best_epoch = 0
    patience_counter = 0
    history = []
    
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            model.zero_grad()
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_loader)
        
        # --- Validation ---
        model.eval()
        y_true, y_pred = [], []
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask=mask)
                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                y_pred.extend(preds)
                y_true.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        mcc = matthews_corrcoef(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average="macro")
        acc = accuracy_score(y_true, y_pred)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Loss: {avg_val_loss:.4f} | MCC: {mcc:.4f} | F1: {f1:.4f}")
        
        history.append({
            "Model": model_name,
            "Epoch": epoch + 1,
            "Val Loss": avg_val_loss,
            "MCC": mcc,
            "F1": f1,
            "Accuracy": acc
        })
        
        # --- Early Stopping & Checkpoint ---
        if mcc > best_mcc:
            best_mcc = mcc
            best_epoch = epoch + 1
            patience_counter = 0
            # Aquí podrías guardar el modelo: torch.save(model.state_dict(), f"{model_name}_best.pt")
        else:
            patience_counter += 1
            
        if patience_counter >= PATIENCE:
            print(f"Early Stopping activado. No mejora desde Epoch {best_epoch}.")
            break
            
    print(f"Mejor MCC para {model_name}: {best_mcc:.4f} (Epoch {best_epoch})")
    return history

# -------------------------------------
# 3. Ejecución y Comparativa
# -------------------------------------
all_results = []

for m in models_to_compare:
    res = train_model(m, X_train, y_train, X_val, y_val)
    all_results.extend(res)

# Crear DataFrame final
df_res = pd.DataFrame(all_results)
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)

# Mostrar la mejor fila de cada modelo (basado en MCC)
best_rows = df_res.loc[df_res.groupby("Model")["MCC"].idxmax()].sort_values("MCC", ascending=False)
print(best_rows[["Model", "Epoch", "Accuracy", "F1", "MCC"]])

Usando dispositivo: cuda
Datos preparados. Train (Balanceado): 18194 | Val: 2379

PROCESANDO: distilbert-base-uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.4026 | MCC: 0.0573 | F1: 0.5276
Epoch 2/10 | Val Loss: 0.4854 | MCC: 0.0628 | F1: 0.5308
Epoch 3/10 | Val Loss: 0.5940 | MCC: 0.0668 | F1: 0.5329
Epoch 4/10 | Val Loss: 0.5596 | MCC: 0.0253 | F1: 0.5124
Epoch 5/10 | Val Loss: 0.5690 | MCC: 0.0790 | F1: 0.5395
Epoch 6/10 | Val Loss: 0.4614 | MCC: 0.0382 | F1: 0.5178
Epoch 7/10 | Val Loss: 0.5046 | MCC: 0.0424 | F1: 0.5194
Epoch 8/10 | Val Loss: 0.4310 | MCC: 0.0609 | F1: 0.5224
Early Stopping activado. No mejora desde Epoch 5.
Mejor MCC para distilbert-base-uncased: 0.0790 (Epoch 5)

PROCESANDO: distilroberta-base


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10 | Val Loss: 0.6459 | MCC: 0.0664 | F1: 0.5031
Epoch 2/10 | Val Loss: 0.5894 | MCC: 0.0808 | F1: 0.5396
Epoch 3/10 | Val Loss: 0.7481 | MCC: 0.1190 | F1: 0.5514
Epoch 4/10 | Val Loss: 0.6199 | MCC: 0.0750 | F1: 0.5373
Epoch 5/10 | Val Loss: 0.5200 | MCC: 0.0719 | F1: 0.5359
Epoch 6/10 | Val Loss: 0.5618 | MCC: 0.0670 | F1: 0.5332
Early Stopping activado. No mejora desde Epoch 3.
Mejor MCC para distilroberta-base: 0.1190 (Epoch 3)

TABLA COMPARATIVA FINAL
                      Model  Epoch  Accuracy        F1       MCC
10       distilroberta-base      3  0.887768  0.551430  0.119035
4   distilbert-base-uncased      5  0.924758  0.539465  0.078980


##### **Tarea 2: predicción del hablante**


### **Otros modelos**


##### **Tarea 1: predicción veracidad**


##### **Tarea 2: predicción del hablante**
